In [1]:
import os
import glob

import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.stats import skew, kurtosis
from scipy.signal import savgol_filter

In [2]:
import sys
sys.path.append("..") 
from pre_processing import  create_vitaldb_feature_set, extract_features_from_dataset

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BASE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/vitaldb_checkpoints"
UCI_PREPROCESSING_DIR = (
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/"
    "UCI/Feature_extraction/Non_fiducial_features/"
    "8sec_50%overlap/data"
)
checkpoint_files = sorted(
    glob.glob(
        os.path.join(
            BASE_DIR,
            "case_*.npz"
        )
    )
)

print(
    "Number of VitalDB case files:",
    len(checkpoint_files)
)

print("\nFirst few files:")
for f in checkpoint_files[:5]:
    print(f)

Number of VitalDB case files: 1995

First few files:
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/vitaldb_checkpoints/case_1.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/vitaldb_checkpoints/case_10.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/vitaldb_checkpoints/case_1001.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/vitaldb_checkpoints/case_1002.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/vitaldb_checkpoints/case_1003.npz


In [4]:
data = np.load(
    checkpoint_files[0]
)

print("Keys:")
print(data.files)

X = data["X"]
y = data["y"]

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nX dtype:", X.dtype)
print("y dtype:", y.dtype)

Keys:
['X', 'y', 'case_id']

X shape: (2481, 1000)
y shape: (2481, 2)

X dtype: float32
y dtype: float32


In [5]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from joblib import Parallel, delayed


# --------------------------------------------------
# Find all cases
# --------------------------------------------------

npz_files = sorted(
    glob.glob(os.path.join(BASE_DIR, "case_*.npz"))
)

print("Number of .npz files:", len(npz_files))


# --------------------------------------------------
# Function to process ONE case
# --------------------------------------------------

def process_case(file):

    data = np.load(file)

    X = data["X"]
    y = data["y"]

    df_case = extract_features_from_dataset(
        X,
        y,
        fs=125,
        n_jobs=1
    )

    case_id = os.path.basename(file).replace(".npz", "")

    df_case["case_id"] = case_id

    return df_case


# --------------------------------------------------
# Parallel processing + tqdm
# --------------------------------------------------

parallel = Parallel(
    n_jobs=8,
    backend="loky",
    return_as="generator_unordered"
)

results = parallel(
    delayed(process_case)(file)
    for file in npz_files
)

# --------------------------------------------------
# Collect results while showing progress
# --------------------------------------------------

all_dfs = []

for df_case in tqdm(
    results,
    total=len(npz_files),
    desc="Extracting features",
    unit="case"
):
    all_dfs.append(df_case)


# --------------------------------------------------
# Combine all cases
# --------------------------------------------------

df_vitaldb = pd.concat(
    all_dfs,
    ignore_index=True
)

print("Final shape:", df_vitaldb.shape)
print(df_vitaldb.head())

Number of .npz files: 1995


Extracting features: 100%|██████████| 1995/1995 [3:42:47<00:00,  6.70s/case]  


Final shape: (5946177, 60)
   ppg_mean  ppg_median   ppg_std  ppg_variance   ppg_iqr  ppg_skewness  \
0  0.472092    0.403251  0.299867      0.089920  0.584228      0.332322   
1  0.475219    0.409882  0.301135      0.090683  0.586067      0.316386   
2  0.478397    0.412583  0.297948      0.088773  0.579901      0.304241   
3  0.477965    0.412628  0.300479      0.090288  0.587720      0.298195   
4  0.468006    0.392603  0.295229      0.087160  0.559894      0.356522   

   ppg_kurtosis  ppg_zero_crossing_rate  ppg_shannon_entropy  ppg_energy_mean  \
0     -1.413868                     0.0             5.392793         0.312791   
1     -1.411997                     0.0             5.424002         0.316516   
2     -1.407996                     0.0             5.425349         0.317636   
3     -1.422631                     0.0             5.441358         0.318738   
4     -1.357200                     0.0             5.462373         0.306189   

   ...  apg_energy_kurtosis  apg_en

In [6]:
VITALDB_SAVE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/data"
os.makedirs(VITALDB_SAVE_DIR, exist_ok=True)

In [7]:
SBP_FTEST_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "FTEST", "SBP")
SBP_MRMR_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "MRMR", "SBP")
SBP_RELIEFF_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "RELIEFF", "SBP")

DBP_FTEST_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "FTEST", "DBP")
DBP_MRMR_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "MRMR", "DBP")
DBP_RELIEFF_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "RELIEFF", "DBP")

Processing: SBP_FTEST
Loading: /data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/8sec_50%overlap/data/SBP_FTEST_preprocessing.pkl
Target: SBP
Number of features: 15
Features:
['vpg_iqr', 'vpg_skewness', 'apg_zero_crossing_rate', 'vpg_energy_iqr', 'apg_energy_iqr', 'apg_skewness', 'vpg_kurtosis', 'ppg_iqr', 'vpg_shannon_entropy', 'apg_shannon_entropy', 'vpg_kte_iqr', 'apg_iqr', 'vpg_energy_skewness', 'ppg_kte_iqr', 'ppg_kurtosis']

VitalDB shape: (5946177, 17)
Saved: /data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/data/SBP_FTEST_vitaldb.csv

Processing: SBP_MRMR
Loading: /data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/8sec_50%overlap/data/SBP_MRMR_preprocessing.pkl
Target: SBP
Number of features: 15
Features:
['ppg_mean', 'vpg_kte_kurtosis', 'ppg_zero_crossing_rate', 'apg_mean', 'vpg_skewness', 'ppg_variance', 'apg_median', 'apg_energy_variance', 'vpg_mean', 'vpg_